# Sweep gen_length — DualCache Baseline on GSM8K

扫描 `gen_length` ∈ {256, 192, 128, 64}，其余参数固定，全量 GSM8K。

> 注：原始 196 不被 block_length=32 整除，已调整为 192。

## 1. 环境设置

In [ ]:
import os, torch, gc

os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3,4,5,6,7'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

os.chdir('llada')
os.makedirs('nlogs', exist_ok=True)

torch.cuda.empty_cache()
gc.collect()

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.2f} GB")

## 2. 四组实验并行启动（4 GPU）

| GPU | gen_length | steps | block_length | threshold |
|-----|-----------|-------|-------------|----------|
| 0   | 256       | 256   | 32          | 0.9      |
| 1   | 192       | 192   | 32          | 0.9      |
| 2   | 128       | 128   | 32          | 0.9      |
| 3   | 64        | 64    | 32          | 0.9      |

In [ ]:
import subprocess, datetime

task = "gsm8k"
fewshot = 5
seed = 42
block_length = 32
threshold = 0.9
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

gen_lengths = [256, 192, 128, 64]

processes = []

for gpu, gl in enumerate(gen_lengths):
    name = f"dualcache_gl{gl}"
    log_file = f"nlogs/sweep_gl_{task}_{name}_{timestamp}.log"
    output_dir = f"evals_results/sweep_gl/{task}-{name}-{timestamp}"
    records_dir = f"evals_results/sweep_gl/{task}-{name}-{timestamp}/step_records"

    model_args_str = ",".join([
        f"model_path='GSAI-ML/LLaDA-8B-Instruct'",
        f"gen_length={gl}",
        f"steps={gl}",
        f"block_length={block_length}",
        f"threshold={threshold}",
        "use_cache=True",
        "show_speed=True",
        f"step_records_dir='{records_dir}'",
        f"seed={seed}",
        "dual_cache=True",
        "mid_block_expand=True",
        "mid_trigger_ratio=0.0",
        "rewarm_on_expand=True",
    ])

    cmd = (
        f"CUDA_VISIBLE_DEVICES={gpu} accelerate launch eval_llada.py "
        f"--tasks {task} --num_fewshot {fewshot} "
        f"--confirm_run_unsafe_code --model llada_dist "
        f"--model_args {model_args_str} "
        f"--output_path {output_dir} --log_samples"
    )

    print(f"GPU {gpu} | gen_length={gl} | {name}")
    print(f"  Log: {log_file}")
    print(f"  Cmd: {cmd[:200]}...")
    print()

    p = subprocess.Popen(cmd, shell=True, stdout=open(log_file, 'w'), stderr=subprocess.STDOUT)
    processes.append((p, name, log_file, output_dir, gl))

print(f"\n{len(processes)} tasks launched in parallel, waiting...")

In [ ]:
for p, name, log, out_dir, gl in processes:
    p.wait()
    rc = p.returncode
    status = "OK" if rc == 0 else f"FAILED (exit {rc})"
    print(f"{name}: {status}")

    with open(log, 'r') as f:
        lines = f.readlines()
    print(f"  Last lines:")
    for line in lines[-15:]:
        print(f"    {line.rstrip()}")
    print()

## 3. 解析结果

In [ ]:
import glob, re, json
import pandas as pd

log_files = sorted(glob.glob(f"nlogs/sweep_gl_{task}_*_{timestamp}.log"))

print(f"Sweep gen_length — DualCache Baseline (timestamp={timestamp})")
print("=" * 100)

rows = []
for log_file in log_files:
    fname = os.path.basename(log_file)
    name_part = fname.replace(f"sweep_gl_{task}_", "").replace(f"_{timestamp}.log", "")

    gl_match = re.search(r'gl(\d+)', name_part)
    gl_val = int(gl_match.group(1)) if gl_match else None

    with open(log_file, 'r') as f:
        content = f.read()

    flex_m = re.search(r'flexible-extract.*?exact_match.*?([\d.]+)', content)
    strict_m = re.search(r'strict-match.*?exact_match.*?([\d.]+)', content)
    speed_m = re.search(r'Tokens per second:\s*([\d.]+)', content)
    nfe_m = re.search(r'Total NFE is (\d+)', content)
    time_m = re.search(r'Total time taken:\s*([\d.]+)', content)
    tok_m = re.search(r'Total number of tokens generated:\s*(\d+)', content)

    rows.append({
        'gen_length': gl_val,
        'flex_acc': float(flex_m.group(1)) if flex_m else None,
        'strict_acc': float(strict_m.group(1)) if strict_m else None,
        'tok_per_sec': float(speed_m.group(1)) if speed_m else None,
        'total_nfe': int(nfe_m.group(1)) if nfe_m else None,
        'total_tokens': int(tok_m.group(1)) if tok_m else None,
        'time_sec': float(time_m.group(1)) if time_m else None,
    })

df = pd.DataFrame(rows).sort_values('gen_length', ascending=False)

print(f"{'gen_length':<12} {'Flex Acc':<10} {'Strict Acc':<12} {'Tok/s':<10} {'NFE':<10} {'Time(s)':<10}")
print("-" * 70)
for _, r in df.iterrows():
    fa = f"{r['flex_acc']:.4f}" if r['flex_acc'] is not None else 'N/A'
    sa = f"{r['strict_acc']:.4f}" if r['strict_acc'] is not None else 'N/A'
    sp = f"{r['tok_per_sec']:.1f}" if r['tok_per_sec'] is not None else 'N/A'
    nf = str(r['total_nfe']) if r['total_nfe'] is not None else 'N/A'
    tm = f"{r['time_sec']:.1f}" if r['time_sec'] is not None else 'N/A'
    print(f"{r['gen_length']:<12} {fa:<10} {sa:<12} {sp:<10} {nf:<10} {tm:<10}")

display(df)